In [1]:
import pandas as pd

complaints = pd.DataFrame([
    {
        "complaint_id": "CMP-001",
        "customer_id": 1002,
        "category": "Billing",
        "description": "Charged extra for data usage",
        "created_at": "2025/09/25 10:45",
        "status": "Open"
    },
    {
        "complaint_id": "CMP-002",
        "customer_id": 1004,
        "category": "Network",
        "description": "Frequent call drops in Delhi",
        "created_at": "2025-09-25 09:30",
        "status": "Open"
    },
    {
        "complaint_id": "CMP-003",
        "customer_id": 1005,
        "category": "Recharge",
        "description": "Recharge failed; amount deducted",
        "created_at": "25-09-2025 14:00",
        "status": "Closed"
    },
    {
        "complaint_id": "CMP-004",
        "customer_id": 1002,
        "category": "Network",
        "description": "Slow 4G speed at night",
        "created_at": "2025-09-26 20:40",
        "status": "Open"
    },
    {
        "complaint_id": "CMP-005",
        "customer_id": 1003,
        "category": "Support",
        "description": "No response to complaint",
        "created_at": "2025-09-26 11:10",
        "status": "Open"
    }
])

complaints.to_csv("complaints.csv", index=False)

print("✅ complaints.csv saved successfully!")

✅ complaints.csv saved successfully!


In [2]:
!pip install pandas sqlalchemy pymysql

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------- ----- 1.8/2.2 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.6 MB/s  0:00:00

   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqla

In [9]:
import pandas as pd
from sqlalchemy import create_engine

user = "root"
password = "admin"       # apna MySQL password
host = "localhost"
database = "testdb"      # IMPORTANT: yahan testdb

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}/{"testdb"}"
)

print("✅ Connected to MySQL successfully!")




✅ Connected to MySQL successfully!


In [12]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse

user = "root"
password = urllib.parse.quote_plus("Rohit@123")  # Encodes '@' to '%40'
host = "localhost"
database = "testdb"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:3306/{database}"
)

try:
    with engine.connect() as conn:
        print("✅ MySQL Connection Successful!")

        result = conn.exec_driver_sql("SELECT DATABASE();")
        print("Database:", result.fetchone()[0])

except Exception as e:
    print("❌ Connection Failed!")
    print(type(e).__name__)
    print(e)

✅ MySQL Connection Successful!
Database: testdb


In [13]:
# Customers from MySQL
customers = pd.read_sql(
    "SELECT * FROM customers",
    engine
)

# Complaints from CSV
complaints = pd.read_csv("complaints.csv")

print("Customers:", len(customers))
print("Complaints:", len(complaints))

display(customers.head())
display(complaints.head())

Customers: 10
Complaints: 5


,customer_id,name,plan_type,join_date,region
0,1001,Asha Mehta,Prepaid,2023-05-12,Delhi
1,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai
2,1003,Sneha Rao,Prepaid,2023-01-18,Chennai
3,1004,Manoj Singh,Postpaid,2021-11-05,Delhi
4,1005,Divya Jain,Prepaid,2023-03-28,Kolkata


,complaint_id,customer_id,category,description,created_at,status
0,CMP-001,1002,Billing,Charged extra for data usage,2025/09/25 10:45,Open
1,CMP-002,1004,Network,Frequent call drops in Delhi,2025-09-25 09:30,Open
2,CMP-003,1005,Recharge,Recharge failed; amount deducted,25-09-2025 14:00,Closed
3,CMP-004,1002,Network,Slow 4G speed at night,2025-09-26 20:40,Open
4,CMP-005,1003,Support,No response to complaint,2025-09-26 11:10,Open


In [14]:
# -----------------------------
# STANDARDIZE TEXT
# -----------------------------

customers["region"] = (
    customers["region"]
    .astype(str)
    .str.title()
    .str.strip()
)

complaints["status"] = (
    complaints["status"]
    .astype(str)
    .str.title()
    .str.strip()
)

complaints["category"] = (
    complaints["category"]
    .astype(str)
    .str.title()
    .str.strip()
)


# -----------------------------
# DATE CONVERSION
# -----------------------------

customers["join_date"] = pd.to_datetime(
    customers["join_date"],
    errors="coerce"
)

complaints["created_at"] = pd.to_datetime(
    complaints["created_at"],
    errors="coerce"
)


# -----------------------------
# FILL MISSING DATES
# -----------------------------

default_dt = pd.Timestamp("2025-09-25 00:00")

customers["join_date"] = customers["join_date"].fillna(default_dt)

complaints["created_at"] = complaints["created_at"].fillna(default_dt)


# -----------------------------
# FIX MISSING VALUES
# -----------------------------

complaints["customer_id"] = (
    complaints["customer_id"]
    .fillna(-1)
    .astype(int)
)

complaints["description"] = (
    complaints["description"]
    .fillna("No description provided")
)


print("✅ Data cleaning completed!")

✅ Data cleaning completed!


In [15]:
merged = customers.merge(
    complaints,
    on="customer_id",
    how="left"
)

print("✅ Data merged successfully!")

display(merged)

✅ Data merged successfully!


,customer_id,name,plan_type,join_date,region,complaint_id,category,description,created_at,status
0,1001,Asha Mehta,Prepaid,2023-05-12,Delhi,NaN,NaN,NaN,NaT,NaN
1,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-001,Billing,Charged extra for data usage,2025-09-25 10:45:00,Open
2,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-004,Network,Slow 4G speed at night,2025-09-25 00:00:00,Open
3,1003,Sneha Rao,Prepaid,2023-01-18,Chennai,CMP-005,Support,No response to complaint,2025-09-25 00:00:00,Open
4,1004,Manoj Singh,Postpaid,2021-11-05,Delhi,CMP-002,Network,Frequent call drops in Delhi,2025-09-25 00:00:00,Open
5,1005,Divya Jain,Prepaid,2023-03-28,Kolkata,CMP-003,Recharge,Recharge failed; amount deducted,2025-09-25 00:00:00,Closed
6,1001,Asha Mehta,Prepaid,2023-05-12,Delhi,NaN,NaN,NaN,NaT,NaN
7,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-001,Billing,Charged extra for data usage,2025-09-25 10:45:00,Open
8,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-004,Network,Slow 4G speed at night,2025-09-25 00:00:00,Open
9,1003,Sneha Rao,Prepaid,2023-01-18,Chennai,CMP-005,Support,No response to complaint,2025-09-25 00:00:00,Open


In [16]:
merged["complaint_id"] = (
    merged["complaint_id"]
    .fillna("NO-COMPLAINT")
)

merged["category"] = (
    merged["category"]
    .fillna("No Complaint")
)

merged["status"] = (
    merged["status"]
    .fillna("Resolved")
)

merged["created_at"] = (
    merged["created_at"]
    .fillna(default_dt)
)


# Open complaint flag
merged["is_open"] = (
    merged["status"] == "Open"
)

print("✅ Transformation completed!")
display(merged)

✅ Transformation completed!


,customer_id,name,plan_type,join_date,region,complaint_id,category,description,created_at,status,is_open
0,1001,Asha Mehta,Prepaid,2023-05-12,Delhi,NO-COMPLAINT,No Complaint,NaN,2025-09-25 00:00:00,Resolved,False
1,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-001,Billing,Charged extra for data usage,2025-09-25 10:45:00,Open,True
2,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-004,Network,Slow 4G speed at night,2025-09-25 00:00:00,Open,True
3,1003,Sneha Rao,Prepaid,2023-01-18,Chennai,CMP-005,Support,No response to complaint,2025-09-25 00:00:00,Open,True
4,1004,Manoj Singh,Postpaid,2021-11-05,Delhi,CMP-002,Network,Frequent call drops in Delhi,2025-09-25 00:00:00,Open,True
5,1005,Divya Jain,Prepaid,2023-03-28,Kolkata,CMP-003,Recharge,Recharge failed; amount deducted,2025-09-25 00:00:00,Closed,False
6,1001,Asha Mehta,Prepaid,2023-05-12,Delhi,NO-COMPLAINT,No Complaint,NaN,2025-09-25 00:00:00,Resolved,False
7,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-001,Billing,Charged extra for data usage,2025-09-25 10:45:00,Open,True
8,1002,Ravi Kumar,Postpaid,2022-12-20,Mumbai,CMP-004,Network,Slow 4G speed at night,2025-09-25 00:00:00,Open,True
9,1003,Sneha Rao,Prepaid,2023-01-18,Chennai,CMP-005,Support,No response to complaint,2025-09-25 00:00:00,Open,True


In [17]:
merged.to_csv(
    "etl_output.csv",
    index=False
)

print(
    f"✅ ETL pipeline complete! "
    f"Created {len(merged)} records."
)

print("📁 File: etl_output.csv")

✅ ETL pipeline complete! Created 12 records.
📁 File: etl_output.csv


In [18]:
print("Complaints per customer:")

result = merged.groupby(
    ["customer_id", "name"]
)["complaint_id"].count().reset_index(
    name="complaint_count"
)

display(result)

Complaints per customer:


,customer_id,name,complaint_count
0,1001,Asha Mehta,2
1,1002,Ravi Kumar,4
2,1003,Sneha Rao,2
3,1004,Manoj Singh,2
4,1005,Divya Jain,2


In [19]:
print("Open vs Closed:")

print(
    merged["status"].value_counts()
)

Open vs Closed:
status
Open        8
Resolved    2
Closed      2
Name: count, dtype: int64


In [20]:
print("Complaints by region & category:")

result = merged.groupby(
    ["region", "category"]
)["complaint_id"].count().reset_index(
    name="count"
)

display(result)

Complaints by region & category:


,region,category,count
0,Chennai,Support,2
1,Delhi,Network,2
2,Delhi,No Complaint,2
3,Kolkata,Recharge,2
4,Mumbai,Billing,2
5,Mumbai,Network,2
